# Multilingual Document OCR + MRZ Parser
## Domino Data Lab — Interactive Notebook

---

### Overview
This notebook performs **OCR on ID cards, passports, residency and bank documents** that may contain Arabic, French, and English text.  
When a **Machine Readable Zone (MRZ)** is detected the notebook additionally parses and validates it against the ICAO 9303 standard using **OmniMRZ**.

### Key technology
| Component | Library | Notes |
|-----------|---------|-------|
| OCR engine | `paddleocr 2.8.1` | PP-OCRv4 mobile, CPU-only |
| ONNX inference | `onnxruntime 1.19.0` | HPI mode – no GPU required |
| MRZ parsing | `omnimrz 0.1.6` | ICAO 9303 validation |
| Excel output | `openpyxl 3.1.5` | Multi-sheet formatted report |

### Prerequisites
1. Upload `requirements.txt` to your Domino project.
2. In **Domino → Environments**, create (or edit) your compute environment and add:  
   ```
   RUN pip install -r /path/to/requirements.txt
   ```
   or paste the packages directly into the Dockerfile instructions.
3. Place document images under `/mnt/data/documents/` (or any path you prefer).
4. Run cells top-to-bottom.

### Expected output
- `/mnt/artifacts/results.json` – raw + structured OCR results per document  
- `/mnt/artifacts/report.xlsx` – formatted Excel report (3 sheets)

---
## 1. Environment & Imports

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import json
import logging
import os
import re
import time
from pathlib import Path
from typing import Any

# ── Third-party ─────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

# ── Logging ─────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)
print('Imports OK')

---
## 2. Configuration
Edit the variables in this cell to match your Domino workspace paths and language preferences.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
INPUT_DIR      = Path('/mnt/data/documents')       # directory or single image
OUTPUT_JSON    = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL   = Path('/mnt/artifacts/report.xlsx')

# ── OCR settings ─────────────────────────────────────────────────────────────
LANGUAGES      = 'ar,fr,en'   # PaddleOCR multilingual code
MAX_DIMENSION  = 1280         # px – resize large images to this max side length
CPU_THREADS    = os.cpu_count() or 4

# ── Supported image extensions ───────────────────────────────────────────────
SUPPORTED_EXT  = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}

# ── MRZ heuristic ────────────────────────────────────────────────────────────
MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

print(f'Config OK  |  threads={CPU_THREADS}  |  lang={LANGUAGES}')

---
## 3. Image Preprocessing Helper

In [ ]:
def load_and_resize(image_path: Path, max_dim: int = MAX_DIMENSION) -> np.ndarray:
    """
    Load an image from disk and downscale if either dimension exceeds *max_dim*.
    Smaller images are returned as-is.
    Resizing speeds up CPU inference while preserving text readability.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read: {image_path}')
    h, w = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)
    if scale < 1.0:
        img = cv2.resize(
            img,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
        log.debug('Resized %s  %dx%d → %dx%d', image_path.name, w, h, img.shape[1], img.shape[0])
    return img

print('load_and_resize defined')

---
## 4. Initialise the Multilingual OCR Engine

> **Note:** First run will download PP-OCRv4 mobile model weights (~30 MB).  
> Subsequent runs use the cached models.

Key options:
- `enable_hpi=True` – activates High-Performance Inference via **ONNX Runtime** (no GPU required)
- `use_angle_cls=False` – skips angle classification; documents are assumed upright
- `ocr_version='PP-OCRv4'` – lightest available model family

In [ ]:
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(
    lang=LANGUAGES,
    use_angle_cls=False,      # documents assumed upright
    use_gpu=False,            # CPU-only
    enable_hpi=True,          # ONNX Runtime HPI
    cpu_threads=CPU_THREADS,
    ocr_version='PP-OCRv4',   # mobile – fast & accurate
    show_log=False,
)

log.info('OCR engine ready  lang=%s  threads=%d', LANGUAGES, CPU_THREADS)

---
## 5. MRZ Detection & OmniMRZ Parsing

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """
    Heuristic scan of OCR text lines for MRZ-like content.

    A line is considered MRZ if it consists solely of uppercase letters,
    digits, and '<' filler characters and is at least 30 characters long.
    Valid ICAO MRZ lines are exactly 30 (TD1), 36 (TD2), or 44 (TD3/MRP)
    characters; we use ≥30 to tolerate minor OCR artefacts.
    Returns a list of cleaned MRZ line strings.
    """
    mrz_lines: list[str] = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')  # OCR may insert spaces
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz_lines.append(cleaned)
    return mrz_lines


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """
    Call OmniMRZ to parse and validate MRZ lines.

    Returns
    -------
    dict with keys:
        raw_mrz       – original line list
        parsed_fields – structured ICAO 9303 fields
        validation    – per-field and overall checksum results
    """
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed – structural MRZ parsing skipped')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}

    mrz_string = '\n'.join(mrz_lines)
    try:
        mrz_obj = MRZ(mrz_string)

        # ── Extract structured fields ────────────────────────────────────
        fields: dict[str, Any] = {
            'surname':          getattr(mrz_obj, 'surname',          None),
            'given_names':      getattr(mrz_obj, 'given_names',      None),
            'document_number':  getattr(mrz_obj, 'document_number',  None),
            'nationality':      getattr(mrz_obj, 'nationality',      None),
            'birth_date':       getattr(mrz_obj, 'birth_date',       None),
            'sex':              getattr(mrz_obj, 'sex',              None),
            'expiry_date':      getattr(mrz_obj, 'expiry_date',      None),
            'document_type':    getattr(mrz_obj, 'document_type',    None),
            'issuing_country':  getattr(mrz_obj, 'issuing_country',  None),
            'optional_data_1':  getattr(mrz_obj, 'optional_data_1', None),
            'optional_data_2':  getattr(mrz_obj, 'optional_data_2', None),
        }

        # ── Validation flags ─────────────────────────────────────────────
        validation: dict[str, Any] = {}
        for attr in dir(mrz_obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try:
                    validation[attr] = getattr(mrz_obj, attr)
                except Exception:
                    pass

        overall = getattr(mrz_obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall

        return {
            'raw_mrz':       mrz_lines,
            'parsed_fields': fields,
            'validation':    validation,
        }

    except Exception as exc:
        log.warning('OmniMRZ parsing failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('MRZ helpers defined')

---
## 6. Single-Document Processing Function

In [ ]:
def process_document(image_path: Path, engine: Any) -> dict[str, Any]:
    """
    Run OCR on one image and optionally parse MRZ.

    Returns
    -------
    dict with keys:
        file_name   – image filename
        full_text   – concatenated OCR lines
        has_mrz     – True if an MRZ was detected
        mrz_data    – parsed MRZ dict or None
        error       – exception message or None
        elapsed_sec – wall-clock processing time
    """
    result: dict[str, Any] = {
        'file_name':   image_path.name,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()
    try:
        img = load_and_resize(image_path)

        # PaddleOCR returns [ [ [box, (text, score)], ... ] ]
        raw = engine.ocr(img, cls=False)

        text_lines: list[str] = []
        if raw and raw[0]:
            for item in raw[0]:
                if item and len(item) >= 2:
                    txt = item[1][0] if isinstance(item[1], (list, tuple)) else str(item[1])
                    text_lines.append(txt)

        result['full_text'] = '\n'.join(text_lines)

        # ── MRZ detection ────────────────────────────────────────────────
        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected  (%d lines)', len(mrz_lines))
        else:
            log.info('  – No MRZ found')

    except Exception as exc:
        log.error('Error processing %s: %s', image_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('process_document defined')

---
## 7. Process a Single Sample Image

Change `SAMPLE_IMAGE` to point to an actual document image in your workspace.  
Skip to **Section 8** if you want to process an entire directory.

In [ ]:
SAMPLE_IMAGE = Path('/mnt/data/documents/sample_passport.jpg')  # ← edit me

if SAMPLE_IMAGE.exists():
    single_result = process_document(SAMPLE_IMAGE, ocr_engine)
    print(json.dumps(single_result, ensure_ascii=False, indent=2, default=str))
else:
    print(f'[SKIP] {SAMPLE_IMAGE} not found – place an image there and re-run.')

---
## 8. Batch Processing

Processes all images found under `INPUT_DIR` (configured in Section 2).

In [ ]:
def collect_images(input_path: Path) -> list[Path]:
    """Return sorted list of image paths from a file or directory."""
    if input_path.is_file():
        return [input_path]
    return sorted(p for p in input_path.rglob('*') if p.suffix.lower() in SUPPORTED_EXT)


images = collect_images(INPUT_DIR)
log.info('Found %d image(s) under %s', len(images), INPUT_DIR)

all_results: list[dict[str, Any]] = []

for img_path in tqdm(images, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', img_path.name)
    r = process_document(img_path, ocr_engine)
    all_results.append(r)
    log.info('  Done in %.2fs', r['elapsed_sec'])

log.info(
    'Batch complete: %d docs | %.1fs total',
    len(all_results),
    sum(r['elapsed_sec'] for r in all_results),
)
print(f'Processed {len(all_results)} document(s)')

---
## 9. Save Results as JSON

In [ ]:
def save_json(results: list[dict[str, Any]], output_path: Path) -> None:
    """Write results list to a pretty-printed, UTF-8 JSON file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    log.info('JSON saved → %s  (%d bytes)', output_path, output_path.stat().st_size)


save_json(all_results, OUTPUT_JSON)

---
## 10. Convert JSON to Formatted Excel

Creates a three-sheet workbook:

| Sheet | Content |
|-------|---------|
| **Summary** | One row per document with key metrics |
| **MRZ Details** | Flattened ICAO 9303 fields for MRZ documents |
| **Full Text** | Raw OCR text for every document |

In [ ]:
def _bold_header_row(ws) -> None:
    """Apply bold font and light-blue fill to the first (header) row."""
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font      = Font(bold=True)
        cell.fill      = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)


def _autofit_columns(ws, max_width: int = 60) -> None:
    """Set column widths based on content length (approximate)."""
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_width)


def build_excel(results: list[dict[str, Any]], output_path: Path) -> None:
    """Build a multi-sheet Excel report from the OCR results list."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    summary_rows, mrz_rows, text_rows = [], [], []

    for r in results:
        mrz_data   = r.get('mrz_data') or {}
        validation = mrz_data.get('validation', {}) if r.get('has_mrz') else {}
        overall    = validation.get('overall_valid', None)

        summary_rows.append({
            'file_name':         r.get('file_name'),
            'has_mrz':           r.get('has_mrz'),
            'overall_mrz_valid': overall,
            'elapsed_sec':       r.get('elapsed_sec'),
            'error':             r.get('error'),
        })

        text_rows.append({
            'file_name': r.get('file_name'),
            'full_text': r.get('full_text'),
        })

        if r.get('has_mrz') and mrz_data:
            pf = mrz_data.get('parsed_fields', {}) or {}
            mrz_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(mrz_data.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'optional_data_1': pf.get('optional_data_1'),
                'optional_data_2': pf.get('optional_data_2'),
                'overall_valid':   validation.get('overall_valid'),
                **{f'valid_{k}': v
                   for k, v in validation.items()
                   if k != 'overall_valid'},
            })

    df_summary = pd.DataFrame(summary_rows)
    df_mrz     = pd.DataFrame(mrz_rows) if mrz_rows else pd.DataFrame(columns=['file_name'])
    df_text    = pd.DataFrame(text_rows)

    with pd.ExcelWriter(str(output_path), engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Summary',     index=False)
        df_mrz.to_excel(    writer, sheet_name='MRZ Details', index=False)
        df_text.to_excel(   writer, sheet_name='Full Text',   index=False)

    wb = load_workbook(str(output_path))
    for ws in wb.worksheets:
        _bold_header_row(ws)
        _autofit_columns(ws)
    wb.save(str(output_path))
    log.info('Excel saved → %s  (%d bytes)', output_path, output_path.stat().st_size)


build_excel(all_results, OUTPUT_EXCEL)
print('Excel report created')

---
## 11. Quick Summary Preview

In [ ]:
import pandas as pd

df_summary = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')
print(f'Total documents : {len(df_summary)}')
print(f'With MRZ        : {df_summary.has_mrz.sum()}')
print(f'MRZ valid       : {df_summary.overall_mrz_valid.sum()}')
print(f'Errors          : {df_summary.error.notna().sum()}')
df_summary.head()

---
## 12. Performance Summary

In [ ]:
times = [r['elapsed_sec'] for r in all_results if r.get('elapsed_sec')]
if times:
    print(f'Documents processed : {len(times)}')
    print(f'Total time          : {sum(times):.1f}s')
    print(f'Mean time / doc     : {sum(times)/len(times):.2f}s')
    print(f'Min / Max           : {min(times):.2f}s / {max(times):.2f}s')

---
## Appendix: Domino Environment Setup

### Option A – Dockerfile snippet (paste into Domino environment)
```dockerfile
# Install system dependencies required by OpenCV
RUN apt-get update && apt-get install -y --no-install-recommends \
    libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*

# Copy and install Python requirements
COPY requirements.txt /tmp/requirements.txt
RUN pip install --no-cache-dir -r /tmp/requirements.txt
```

### Option B – Pre-run script
In **Domino → Project Settings → Pre-run script**, add:
```bash
pip install paddleocr==2.8.1 paddlepaddle==2.6.1 onnxruntime==1.19.0 \
    omnimrz==0.1.6 opencv-python-headless==4.10.0.84 Pillow==10.4.0 \
    numpy==1.26.4 pandas==2.2.2 openpyxl==3.1.5 tqdm==4.66.5
```

### Workspace layout
```
/mnt/data/documents/    ← place input images here
/mnt/artifacts/         ← outputs (results.json, report.xlsx) written here
/mnt/code/              ← this notebook and the .py script
```